<a href="https://colab.research.google.com/github/HaniaArif/Document_Color_Classifier/blob/main/document_color_extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install PyPDF2
!pip install pdf2image
!apt-get install poppler-utils
!pip install ghostscript
!pip install camelot-py[cv]
!pip install excalibur-py
!apt install ghostscript python3-tk
!pip install opencv-python
!pip install numpy

In [ ]:
from PIL import Image
from PyPDF2 import PdfFileReader, PdfFileWriter, generic

import sys
import locale
import ghostscript
import os, glob

import cv2
import numpy as np

from google.colab import files

import warnings
warnings.filterwarnings("ignore")

results = {}

#-------------READ PDF FILE-----------------

uploaded = files.upload()
if (uploaded != {}):
  for fn in uploaded.keys():
    input = PdfFileReader(open('/content/'+fn, "rb"))
    #pdf_fp = "/content/drive/MyDrive/full_color/7_3.pdf"
    pdf_fp = '/content/'+fn
    input = PdfFileReader(open(pdf_fp, "rb"))

    #-------------GET PDF PAGE TO BE CHECKED-----------------

    page = input.getPage(0)

    #-------------REMOVE TOMBO-----------------

    (x, y) = page.cropBox.getUpperRight()
    #get center of pdf
    x = int(x/2)
    y = int(y/2)
    #dimensions after cropping as in pdf file 91mmx55mm
    t_x = 270
    t_y = 170
    #half the dimensions
    t_x = t_x/2
    t_y = t_y/2
    #find the starting and ending co-ordinates of area to be cropped w.r.t the center of pdf
    x1 = x - t_x
    y1 = y - t_y
    x2 = x + t_x
    y2 = y + t_y
    #crop pdf
    page.mediaBox.upperLeft = (x1,y1)
    page.mediaBox.lowerRight = (x2,y2)
    #save pdf with no tombo as new file
    output = PdfFileWriter()
    output.addPage(page)
    outputStream = open('result.pdf','wb')
    output.write(outputStream)
    outputStream.close()

    #-------------separate CMYK channels of PDF(with no tombo)-----------------

    os.system('gs -sDEVICE=tiffsep -dBATCH -dNOPAUSE -dNOSUBSTDEVICECOLORS -dSimulateOverprint=true -sProcessColorModel=DeviceCMYK -sColorConversionStrategy=CMYK -sOutputFile=temp_ result.pdf')

    #-------------Channel value correction-----------------

    #load the channels
    c_img = Image.open('temp_(Cyan).tif')
    m_img = Image.open('temp_(Magenta).tif')
    y_img = Image.open('temp_(Yellow).tif')
    k_img = Image.open('temp_(Black).tif')

    #convert channels to numpy arrays for calculations
    try:
      cArray = np.asarray(c_img)
      mArray = np.asarray(m_img)
      yArray = np.asarray(y_img)
      kArray = np.asarray(k_img)
    except:
      print ("ERROR: Converting to numpy array...")
      exit(1)

    #invert channel values
    try:
      toSub = 255
      cInvArray = toSub - cArray
      mInvArray = toSub - mArray
      yInvArray = toSub - yArray
      kInvArray = toSub - kArray
    except:
      print ("ERROR: inverting !")

    #-------------PDF COLOR DETECTION-----------------

    layers = sum('temp_' in s for s in os.listdir("/content/"))
    if layers >=  6:  #special color
      results[fn] = "FULL COLOR PDF WITH SPECIAL COLOR"
      #print("FULL COLOR PDF WITH SPECIAL COLOR")

    #layers are 5 >> PDF with cmyk colors with no special color

    #check monochrome
    elif (np.all(cInvArray == 0) and np.all(mInvArray == 0) and np.all(yInvArray == 0) and np.all(kInvArray >= 0)):
      #all other channels are empty except the k channel
      results[fn] = "MONOCHROME PDF"
      #print("MONOCHROME PDF")

    else:
      #check special color
      toSub1 = 25
      toSub2 = 50
      cSubArray = abs(toSub1 - np.round(cInvArray/255*100))
      mSubArray = abs(toSub1 - np.round(mInvArray/255*100))
      ySubArray = abs(toSub1 - np.round(yInvArray/255*100))
      kSubArray = abs(toSub2 - np.round(kInvArray/255*100))
      if (np.all(cSubArray == 25) and np.all(mSubArray == 25) and np.all(ySubArray == 25) and np.all(kSubArray == 50)):
        results[fn] = "RICK BLACK PDF"
        #print("RICK BLACK PDF")
        #print("DO YOU WANT TO PRINT IN FULL COLOR OR MONOCHROME?")

      #full color pdf
      else:
        results[fn] = "FULL COLOR PDF"
        #print("FULL COLOR PDF")

    #remove all previous layer files
    for filename in glob.glob("/content/temp_*"):
      os.remove(filename)
    #remove temporary pdf file without tombo
    os.remove("result.pdf")
    os.remove(pdf_fp)
  print()
  print()
  print("\033[1mResults: \033[0;0m")
  print('\u2500' * 100)
  print("\033[1m{:<46} {:<10} \033[0;0m".format('File','Color Result'))
  print('\u2500' * 100)
  for k, v in results.items():
    print("{:<50} {:<15}".format(k, v))
else:
  print("NO FILE SELECTED")